# 14 — Text Feature Engineering
**Goal:** Extraer señales numéricas del texto que van más allá de la bolsa de palabras.

```
Features de superficie  → longitud, caps, puntuación, emojis
Features léxicas        → proporción de palabras positivas/negativas
Features de legibilidad → complejidad del texto
Features de estilo      → tipo gramatical aproximado (POS heurístico)
```

Estas features complementan TF-IDF y en muchos casos mejoran el F1 significativamente.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import cross_val_score
from scipy.sparse import hstack, csr_matrix

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
})

STOPWORDS_ES = [
    'a','al','como','con','de','del','el','en','es','la','las','lo','los',
    'me','mi','muy','no','o','para','por','que','se','si','sin','su','te',
    'todo','un','una','y','yo',
]

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\wáéíóúüñ\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
# Comentarios con variación de estilo para ilustrar features
raw_comments = [
    ('LLEVO 3 DÍAS SIN RESPUESTA!!! PÉSIMO SERVICIO 😤😤😤', 0),
    ('Excelente servicio, aprobaron mi tarjeta rápidamente :)', 1),
    ('no me llegó el otp', 0),
    ('La app es MUY lenta... tardé 20 min solo para entrar', 0),
    ('Increíbles beneficios! Recomiendo 100% la tarjeta!!!', 1),
    ('falla todo el tiempo horrible', 0),
    ('La atención al cliente fue excepcional, gracias', 1),
    ('POR QUÉ RECHAZARON MI SOLICITUD SIN EXPLICACIÓN?????', 0),
    ('proceso sencillo y rápido, muy contento', 1),
    ('No funciona el sistema, esto es inaceptable', 0),
] * 30

import random; random.seed(42)
random.shuffle(raw_comments)

df = pd.DataFrame(raw_comments, columns=['comment', 'sentiment'])
print(f'Dataset: {len(df)} comentarios')

## 1. Features de superficie

In [ ]:
def surface_features(text):
    """Features basadas en apariencia del texto, sin NLP."""
    n_chars    = len(text)
    n_words    = len(text.split())
    n_upper    = sum(1 for c in text if c.isupper())
    n_excl     = text.count('!')
    n_quest    = text.count('?')
    n_dots     = text.count('.')
    n_ellipsis = text.count('...')
    n_digits   = sum(1 for c in text if c.isdigit())

    # Emojis: todo lo que no es ASCII ni español
    n_emoji = len(re.findall(r'[^\x00-\x7Fáéíóúüñ\s\w]', text))

    # URLs y hashtags
    n_url  = len(re.findall(r'https?://\S+', text))
    n_hash = len(re.findall(r'#\w+', text))

    # Proporción de mayúsculas
    pct_upper = n_upper / max(n_chars, 1)

    # Longitud media de palabras
    words = text.split()
    avg_word_len = np.mean([len(w) for w in words]) if words else 0

    return {
        'n_chars': n_chars, 'n_words': n_words,
        'n_upper': n_upper, 'pct_upper': pct_upper,
        'n_excl': n_excl, 'n_quest': n_quest,
        'n_dots': n_dots, 'n_ellipsis': n_ellipsis,
        'n_digits': n_digits, 'n_emoji': n_emoji,
        'n_url': n_url, 'n_hash': n_hash,
        'avg_word_len': avg_word_len,
    }

# Calcular para algunos ejemplos
for text, sent in raw_comments[:5]:
    feats = surface_features(text)
    print(f'["{text[:50]}..." | sent={sent}]')
    key_feats = {k: v for k, v in feats.items() if v > 0}
    print(f'  {key_feats}\n')

## 2. Features léxicas — proporción de palabras con polaridad

In [ ]:
LEXICON_POS = set(['excelente', 'increíble', 'perfecto', 'rápido', 'rápidamente',
                   'fácil', 'sencillo', 'recomiendo', 'satisfecho', 'excepcional',
                   'beneficios', 'feliz', 'contento', 'gracias', 'agil'])
LEXICON_NEG = set(['lenta', 'lento', 'falla', 'pésimo', 'terrible', 'ineficiente',
                   'inaceptable', 'horrible', 'rechazaron', 'espera', 'días',
                   'nadie', 'mal', 'problema', 'error', 'fallo'])
NEGATORS    = set(['no', 'sin', 'nunca', 'jamás', 'ni'])

def lexical_features(text):
    tokens = preprocess(text).split()
    n      = max(len(tokens), 1)

    n_pos   = sum(1 for t in tokens if t in LEXICON_POS)
    n_neg   = sum(1 for t in tokens if t in LEXICON_NEG)
    n_neg_w = sum(1 for t in tokens if t in NEGATORS)

    # Detectar negaciones inversoras: 'no' seguido de positivo
    n_negated_pos = 0
    for i, t in enumerate(tokens):
        if t in NEGATORS and i+1 < len(tokens) and tokens[i+1] in LEXICON_POS:
            n_negated_pos += 1

    return {
        'n_pos_words':   n_pos,
        'n_neg_words':   n_neg,
        'pct_pos':       n_pos / n,
        'pct_neg':       n_neg / n,
        'polarity_ratio': (n_pos - n_neg) / (n_pos + n_neg + 1),
        'n_negators':    n_neg_w,
        'n_negated_pos': n_negated_pos,
    }

for text, sent in raw_comments[:5]:
    f = lexical_features(text)
    print(f'sent={sent} | polarity_ratio={f["polarity_ratio"]:+.2f} | {text[:50]}')

## 3. Features de legibilidad

In [ ]:
VOWELS = set('aeiouáéíóúü')

def count_syllables(word):
    """Aproximación: contar grupos de vocales consecutivas."""
    word = word.lower()
    count = 0
    prev_vowel = False
    for c in word:
        if c in VOWELS:
            if not prev_vowel:
                count += 1
            prev_vowel = True
        else:
            prev_vowel = False
    return max(count, 1)

def readability_features(text):
    tokens = preprocess(text).split()
    if not tokens:
        return {'avg_syllables': 0, 'pct_complex_words': 0, 'flesch_approx': 0}

    syllables = [count_syllables(t) for t in tokens]
    avg_syl   = np.mean(syllables)
    n_complex  = sum(1 for s in syllables if s >= 3)  # palabras con ≥3 sílabas
    pct_complex = n_complex / len(tokens)

    # Flesch-Kincaid adaptado al español (aproximado)
    n_sentences = max(len(re.split(r'[.!?]', text)), 1)
    flesch = 206.84 - 1.02 * (len(tokens)/n_sentences) - 60 * avg_syl

    return {
        'avg_syllables':    avg_syl,
        'pct_complex_words': pct_complex,
        'flesch_approx':    flesch,
        'n_sentences':      n_sentences,
        'words_per_sentence': len(tokens) / n_sentences,
    }

test_sentences = [
    'falla todo',
    'El proceso de evaluación crediticia es demasiado lento e ineficiente',
    'Excelente',
]
for s in test_sentences:
    f = readability_features(s)
    print(f'"{s}"')
    print(f'  Flesch≈{f["flesch_approx"]:.0f} | avg_syl={f["avg_syllables"]:.2f} | words/sent={f["words_per_sentence"]:.1f}\n')

## 4. Combinar todo — custom sklearn Transformer

In [ ]:
class TextFeatureExtractor(BaseEstimator, TransformerMixin):
    """sklearn-compatible transformer que extrae features numéricas del texto."""

    def fit(self, X, y=None):
        return self   # nada que aprender

    def transform(self, X):
        rows = []
        for text in X:
            f = {}
            f.update(surface_features(text))
            f.update(lexical_features(text))
            f.update(readability_features(text))
            rows.append(f)
        return pd.DataFrame(rows).fillna(0).to_numpy(dtype=float)

    def get_feature_names_out(self):
        sample = surface_features('test')
        sample.update(lexical_features('test'))
        sample.update(readability_features('test'))
        return list(sample.keys())

# Verificar
fte = TextFeatureExtractor()
X_manual = fte.fit_transform(df['comment'])
print(f'Shape features manuales: {X_manual.shape}')
print(f'Feature names: {fte.get_feature_names_out()}')

## 5. Comparar — TF-IDF solo vs TF-IDF + features manuales

In [ ]:
X_texts = df['comment'].tolist()
y       = df['sentiment'].to_numpy()

# 1. Solo TF-IDF
pipe_tfidf = Pipeline([
    ('tfidf', TfidfVectorizer(preprocessor=preprocess, stop_words=STOPWORDS_ES,
                               ngram_range=(1,2), sublinear_tf=True, min_df=2)),
    ('clf',   LogisticRegression(max_iter=1000)),
])

# 2. Solo features manuales
from sklearn.pipeline import make_pipeline
pipe_manual = make_pipeline(
    TextFeatureExtractor(),
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)

# 3. TF-IDF + features manuales (concatenar sparse + dense)
class CombinedFeatures(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.tfidf = TfidfVectorizer(preprocessor=preprocess, stop_words=STOPWORDS_ES,
                                      ngram_range=(1,2), sublinear_tf=True, min_df=2)
        self.fte   = TextFeatureExtractor()
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        self.tfidf.fit(X)
        X_manual = self.fte.fit_transform(X)
        self.scaler.fit(X_manual)
        return self

    def transform(self, X):
        X_tfidf  = self.tfidf.transform(X)
        X_manual = self.scaler.transform(self.fte.transform(X))
        return hstack([X_tfidf, csr_matrix(X_manual)])

pipe_combined = Pipeline([
    ('features', CombinedFeatures()),
    ('clf', LogisticRegression(max_iter=1000)),
])

print(f'{"Pipeline":<35} {"F1 CV":>8}')
print('-' * 45)
for name, pipe in [
    ('Solo TF-IDF',                  pipe_tfidf),
    ('Solo features manuales',       pipe_manual),
    ('TF-IDF + features manuales',   pipe_combined),
]:
    f1 = cross_val_score(pipe, X_texts, y, cv=5, scoring='f1').mean()
    print(f'{name:<35} {f1:.4f}')

## 6. Importancia de cada feature manual

In [ ]:
# Entrenar solo con features manuales y ver coeficientes
fte.fit(X_texts)
X_manual_all = fte.transform(X_texts)
X_scaled     = StandardScaler().fit_transform(X_manual_all)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_scaled, y)

feat_names = fte.get_feature_names_out()
coefs      = lr.coef_[0]
order      = np.argsort(np.abs(coefs))[::-1][:15]

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#06d6a0' if c > 0 else '#f72585' for c in coefs[order][::-1]]
ax.barh(np.array(feat_names)[order][::-1], coefs[order][::-1], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Importancia de features manuales — LR coeficientes\n(verde=positivo, rojo=negativo)')
ax.set_xlabel('Coeficiente (estandarizado)')
plt.tight_layout()
plt.show()

## Resumen
| Categoría | Features |
|---|---|
| Superficie | longitud, % mayúsculas, exclamaciones, emojis, URLs |
| Léxicas | proporción pos/neg, ratio polaridad, negadores |
| Legibilidad | sílabas promedio, % palabras complejas, Flesch |
| Estilo | palabras por oración, diversidad léxica |

| Patrón | Cuándo usar |
|---|---|
| `BaseEstimator + TransformerMixin` | Custom transformer compatible con sklearn Pipeline |
| `hstack([sparse, dense])` | Concatenar TF-IDF (sparse) con features numéricas (dense) |
| `StandardScaler` | Siempre escalar features numéricas antes de clasificadores lineales |

**Siguiente:** `15_attention_numpy.ipynb` — mecanismo de atención desde cero con NumPy.